###### Notebook 1 — 03_Load_Silver_to_Gold_Augment.py

In [0]:
# ── Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# ✅ M5 FIX: Use %run instead of exec() for config
%run ../config/pipeline_config

In [0]:
# ✅ M5 FIX: Use %run instead of exec() for metrics_query
%run ../utils/metrics_query

In [0]:
mq = MetricsQuery(spark)
print(":white_check_mark: MetricsQuery loaded")

In [0]:
%run /Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline/utils/cdc_validator

In [0]:
# ✅ M5 FIX: Use %run instead of exec() for audit_logger
%run ../utils/audit_logger

In [0]:
from pyspark.sql.functions import col

def get_latest_change_version_and_cdc_count(catalog_name, schema_name, table_name):
    """
    Finds the latest version of a Delta table where a merge, update, insert, or delete operation occurred,
    and returns the count of CDC records since that version.
    """
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    try:
        history_df = spark.sql(f"DESCRIBE HISTORY {full_table_name}")
        relevant_ops = ["MERGE", "UPDATE", "INSERT", "DELETE"]
        filtered_df = history_df.filter(col("operation").isin(relevant_ops))
        latest_version_row = filtered_df.orderBy(col("version").desc()).limit(1).collect()
        if latest_version_row:
            latest_version = latest_version_row[0]["version"]
            #print(f"Latest version on table  {full_table_name} is : {latest_version}")
            # Check if CDC is enabled
            tbl_props = spark.sql(f"SHOW TBLPROPERTIES {full_table_name}")
            cdc_enabled = tbl_props.filter(col("key") == "delta.enableChangeDataFeed").collect()
            if cdc_enabled and cdc_enabled[0]["value"].lower() == "true":
                cdc_df = spark.read.format("delta").option("readChangeData", "true") \
                    .option("startingVersion", latest_version).table(full_table_name)
                cdc_count = cdc_df.count()
                print(f"{full_table_name} CDC ENABLED , Changed records : {cdc_count}")
            else:
                print(f"{full_table_name} CDC DISABLED , Changed records : N/A")
        else:
            print(f"{full_table_name} — No MERGE/UPDATE/INSERT/DELETE operations found in history.")
    except Exception as e:
        print(f"Error reading CDC for {full_table_name}: {e}")

get_latest_change_version_and_cdc_count('hgi','silver','accounts')
get_latest_change_version_and_cdc_count('hgi','silver','contacts')
get_latest_change_version_and_cdc_count('hgi','silver','events')

In [0]:

#NOTEBOOK 1 WITH 3MODULES applied from utils folder(03_Load_Silver_to_Gold_Augment.py)

from pyspark.sql.functions import col, lit, lower, when, current_timestamp, coalesce, split
from delta.tables import DeltaTable
import time

# ===================================================================================
# CONFIGURATION
# ===================================================================================
dbutils.widgets.text("customer_id", "cust_005")
customer_id = dbutils.widgets.get("customer_id").lower()
tenant_id   = tenant_id_from_customer(customer_id)   # TENANT FILTER

load_type = "historical"  # hardcoded — change to "historical" if full backfill needed

# ===================================================================================
# FUTURE ENHANCEMENT — Multi-Customer Loop (uncomment when ready):
#
# customer_ids_df = spark.sql("""
#     SELECT DISTINCT customer_id
#     FROM ingestion_metadata.customers
#     WHERE is_active = true
# """)
# customer_ids = [row.customer_id for row in customer_ids_df.collect()]
# for customer_id in customer_ids:
#     pass
# ===================================================================================

SILVER_CATALOG     = "hgi"
SILVER_SCHEMA      = "silver"
GOLD_CATALOG       = "hgi"
GOLD_SCHEMA        = "gold"
ENRICHMENT_CATALOG = "hgi"
ENRICHMENT_SCHEMA  = "enrichment"
STALE_DAYS         = 180

# ✅ M7 FIX: Use FREE_EMAIL_DOMAINS from pipeline_config (loaded via %run above)
# FREE_EMAIL_DOMAINS = [...]  # REMOVED - already loaded from pipeline_config

# ✅ M6 FIX: Let AQE handle shuffle partitions dynamically
# spark.conf.set("spark.sql.shuffle.partitions", 4)  # REMOVED - let AQE optimize

print("=" * 70)
print("03_Load_Silver_to_Gold_Augment")
print(f"  Customer : {customer_id}")
print(f"  Tenant   : {tenant_id}")                    # TENANT FILTER
print(f"  Load Type: {load_type}")
print("=" * 70)

# CDF_CHECK — list to accumulate all counts, written to table at the end
cdf_counts = []  # CDF_CHECK

# CDF_CHECK — helper to record a count entry
def record_count(stage, table_name, count):          # CDF_CHECK
    cdf_counts.append({                              # CDF_CHECK
        "stage"      : stage,                        # CDF_CHECK
        "table_name" : table_name,                   # CDF_CHECK
        "row_count"  : count,                        # CDF_CHECK
        "customer_id": customer_id                   # CDF_CHECK
    })                                               # CDF_CHECK


# ===================================================================================
# MODULE 1 — METRICS QUERY (START)
# ===================================================================================

#%run ../../utils/metrics_query

#print("✅ MetricsQuery loaded")


# ===================================================================================

spark.sql("ALTER TABLE hgi.silver.contacts_all SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql("ALTER TABLE hgi.silver.accounts_all SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql("ALTER TABLE hgi.silver.contacts SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql("ALTER TABLE hgi.silver.events SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")






#=============================================================================
# MODULE 2 — CDC VALIDATOR (START)
# Validates CDF is enabled on silver input tables before processing.
# Writes validation results to hgi.silver.logs.cdc_validation_log
# ===================================================================================
#%run "../../utils/cdc_validator"

cdc_silver_result = validate_cdc(
    layer       = "silver",
    table_names = ["contacts_all", "contacts", "events", "accounts_all"],
    catalog     = SILVER_CATALOG,
    schema      = SILVER_SCHEMA
)
print(f"  CDC Silver: {cdc_silver_result['message']}")


# ===================================================================================
# MODULE 3 — AUDIT LOGGER (START)
# ===================================================================================
#%run "../../utils/audit_logger"
initialize_audit_tables()
print("✅ AuditLogger loaded and tables initialized")

# ===================================================================================
# CDF_CHECK — CREATE cdf_checker TABLE
# ===================================================================================
# spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.cdf_checks")  # CDF_CHECK

# spark.sql("""                                               
#     CREATE TABLE IF NOT EXISTS bronze.cdf_checks.cdf_checker (
#         run_timestamp TIMESTAMP,
#         customer_id   STRING,
#         stage         STRING,
#         table_name    STRING,
#         row_count     LONG
#     ) USING DELTA
# """)                                                        # CDF_CHECK
# print("✅ [CDF_CHECK] bronze.cdf_checks.cdf_checker ready") # CDF_CHECK


# ===================================================================================
# GOLD TABLE INITIALIZATION
# ===================================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")

def initialize_gold_tables():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.persons (
            mk_email              STRING    COMMENT 'Lookup key lowercase email',
            mk_domain             STRING,
            mk_status             STRING,
            mk_timestamp          TIMESTAMP,
            name__fullname        STRING,
            name__givenname       STRING,
            name__familyname      STRING,
            employment__name      STRING,
            employment__domain    STRING,
            employment__role      STRING,
            employment__seniority STRING,
            employment__title     STRING,
            geo__city             STRING,
            geo__country          STRING,
            geo__state            STRING,
            is_student            BOOLEAN,
            is_spam               BOOLEAN,
            linkedin__handle      STRING,
            twitter__handle       STRING,
            updated_at            TIMESTAMP
        ) USING DELTA CLUSTER BY (mk_email)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.companies (
            mk_domain               STRING    COMMENT 'Lookup key lowercase domain',
            mk_status               STRING,
            mk_timestamp            TIMESTAMP,
            name                    STRING,
            description             STRING,
            category__industry      STRING,
            category__sector        STRING,
            category__industrygroup STRING,
            category__subindustry   STRING,
            metrics__employees      LONG,
            metrics__employeesrange STRING,
            metrics__raised         LONG,
            metrics__marketcap      STRING,
            geo__city               STRING,
            geo__country            STRING,
            geo__state              STRING,
            tech                    STRING,
            tags                    STRING,
            founded_year            LONG,
            emailprovider           BOOLEAN,
            site_url                STRING,
            alexa__globalrank       LONG,
            is_personal             BOOLEAN,
            updated_at              TIMESTAMP
        ) USING DELTA CLUSTER BY (mk_domain)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations (
            contact_id                        STRING,
            tenant                           LONG,
            email                            STRING,
            domain                           STRING,
            person__mk_email                 STRING,
            person__mk_domain                STRING,
            person__mk_status                STRING,
            person__mk_timestamp             TIMESTAMP,
            person__name__fullname           STRING,
            person__name__givenname          STRING,
            person__name__familyname         STRING,
            person__employment__name         STRING,
            person__employment__domain       STRING,
            person__employment__role         STRING,
            person__employment__seniority    STRING,
            person__employment__title        STRING,
            person__geo__city                STRING,
            person__geo__country             STRING,
            person__geo__state               STRING,
            person__is_student               BOOLEAN,
            person__is_spam                  BOOLEAN,
            person__linkedin__handle         STRING,
            person__twitter__handle          STRING,
            company__mk_domain               STRING,
            company__mk_status               STRING,
            company__mk_timestamp            TIMESTAMP,
            company__name                    STRING,
            company__description             STRING,
            company__category__industry      STRING,
            company__category__sector        STRING,
            company__category__industrygroup STRING,
            company__category__subindustry   STRING,
            company__metrics__employees      LONG,
            company__metrics__employeesrange STRING,
            company__metrics__raised         LONG,
            company__metrics__marketcap      STRING,
            company__geo__city               STRING,
            company__geo__country            STRING,
            company__geo__state              STRING,
            company__tech                    STRING,
            company__tags                    STRING,
            company__founded_year            LONG,
            company__emailprovider           BOOLEAN,
            company__site_url                STRING,
            company__alexa__globalrank       LONG,
            company__is_personal             BOOLEAN,
            updated_at                       TIMESTAMP
        ) USING DELTA CLUSTER BY (contact_id)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

initialize_gold_tables()
print("Gold tables initialized.")

# ===================================================================================
# WATERMARK HELPERS
# ===================================================================================
def get_watermark(src_sys, obj_name):
    try:
        result = spark.sql(f"""
            SELECT last_processed_timestamp
            FROM ingestion_metadata.watermarks
            WHERE source_system = '{src_sys}' AND object_name = '{obj_name}'
        """).collect()
        if result:
            return result[0]["last_processed_timestamp"]
    except:
        return None
    return None